# Cinema Backend — Kaggle CPU Phase 2 Evaluation (`load_test_v2`)

Import file này lên Kaggle (**Accelerator = None**, **Internet = On**) rồi bấm **Run All**. Hỗ trợ đo đơn lẻ (`AFTER_REF = None`) hoặc đo đối chứng **Before vs After** ngay trong cùng 1 phiên 4-vCPU.

In [ ]:
# Ô 1 — Cấu hình phiên đo
REPO_URL   = 'https://github.com/HUY-TD16/SA_Phase2_Optimise.git'
SUITE_REF  = 'huy'       # Nhánh chứa thư mục load_test_v2 đã push lên GitHub ('huy' hoặc 'main')
BEFORE_REF = '21eec61'   # Commit Phase 1 dùng làm mốc Before
AFTER_REF  = None        # Điền nhánh/commit Phase 2 (vd: 'main') khi muốn tự động so sánh Before vs After

QUICK_SMOKE_MODE = False # Đặt True nếu muốn chạy thử nhanh toàn bộ trong ~3 phút
SCENARIOS = ('k1', 'k2', 'k3', 'k4', 'k5', 'k6')


In [ ]:
%%bash
set -euo pipefail
sudo apt-get update -qq
sudo apt-get install -y openjdk-21-jdk-headless postgresql postgresql-contrib redis-server nginx nodejs curl ca-certificates gnupg
if ! command -v k6 >/dev/null 2>&1; then
  curl -fsSL https://dl.k6.io/key.gpg | sudo gpg --batch --yes --dearmor -o /usr/share/keyrings/k6-archive-keyring.gpg
  echo 'deb [signed-by=/usr/share/keyrings/k6-archive-keyring.gpg] https://dl.k6.io/deb stable main' | sudo tee /etc/apt/sources.list.d/k6.list
  sudo apt-get update -qq && sudo apt-get install -y k6
fi


In [ ]:
# Ô 3 — Clone repo & sao lưu load_test_v2
import glob, json, os, pathlib, shutil, subprocess, time, urllib.request

jdk = sorted(glob.glob('/usr/lib/jvm/java-21-openjdk-*'))
assert jdk, 'Chưa cài được JDK 21'
os.environ['JAVA_HOME'] = jdk[0]
os.environ['PATH'] = jdk[0] + '/bin:' + os.environ['PATH']

work_root = pathlib.Path('/kaggle/working')
repo = work_root / 'SA_Phase2_Optimise'
if not repo.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', 'checkout', '--force', SUITE_REF], check=True)
assert (repo / 'load_test_v2').exists(), f'Chưa có load_test_v2 trên nhánh {SUITE_REF}! Hãy git push trước.'

suite_backup = work_root / 'suite_v2'
if suite_backup.exists(): shutil.rmtree(suite_backup)
shutil.copytree(repo / 'load_test_v2', suite_backup)
print('Suite ready:', suite_backup)


In [ ]:
# Ô 4 — Khởi động PostgreSQL & Redis
pg_version, pg_name, pg_port = subprocess.check_output(['pg_lsclusters', '--no-header'], text=True).splitlines()[-1].split()[:3]
subprocess.run(['sudo', 'pg_ctlcluster', '--skip-systemctl-redirect', pg_version, pg_name, 'start'], check=False)
subprocess.run(['redis-server', '--daemonize', 'yes', '--port', '6379', '--maxmemory', '256mb', '--maxmemory-policy', 'allkeys-lru'], check=True)
redis_pid = int(subprocess.check_output(['pgrep', '-f', 'redis-server'], text=True).splitlines()[0])

def pg_root(sql):
    return subprocess.check_output(['sudo', '-u', 'postgres', 'psql', '-p', pg_port, '-v', 'ON_ERROR_STOP=1', '-tAc', sql], text=True).strip()

pg_root("DO $$ BEGIN IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname='cinema_load') THEN CREATE ROLE cinema_load LOGIN PASSWORD 'cinema_load_pw'; END IF; END $$;")
database_pid = int(subprocess.check_output(['sudo', 'head', '-n', '1', pg_root('SHOW data_directory') + '/postmaster.pid'], text=True))

os.environ.update({
    'PGHOST': '127.0.0.1', 'PGPORT': pg_port, 'PGDATABASE': 'cinema_load', 'PGUSER': 'cinema_load', 'PGPASSWORD': 'cinema_load_pw',
    'DB_URL': f'jdbc:postgresql://127.0.0.1:{pg_port}/cinema_load', 'DB_USERNAME': 'cinema_load', 'DB_PASSWORD': 'cinema_load_pw',
    'REDIS_HOST': '127.0.0.1', 'REDIS_PORT': '6379',
    'JWT_SECRET': 'kaggle-phase2-cinema-benchmark-secret-key-at-least-32-bytes', 'JWT_EXPIRATION_SECONDS': '14400',
    'DEMO_ADMIN_USERNAME': 'admin', 'DEMO_ADMIN_PASSWORD': 'demo-pass-1234', 'ADMIN_USERNAME': 'admin', 'ADMIN_PASSWORD': 'demo-pass-1234',
    'SPRING_PROFILES_ACTIVE': 'local',
})
print('PostgreSQL:', pg_root('SELECT version()'), '| DB PID:', database_pid, '| Redis PID:', redis_pid)


In [ ]:
# Ô 5 — Hàm build, seed và chạy đo tự động
def run_variant(git_ref, label, jvm_flags, strict=False):
    subprocess.run(['git', 'checkout', '--force', git_ref], check=True)
    if (repo / 'load_test_v2').exists(): shutil.rmtree(repo / 'load_test_v2')
    shutil.copytree(suite_backup, repo / 'load_test_v2')
    pg_root("SELECT pg_terminate_backend(pid) FROM pg_stat_activity WHERE datname='cinema_load' AND pid <> pg_backend_pid();")
    subprocess.run(['sudo', '-u', 'postgres', 'dropdb', '-p', pg_port, '--if-exists', 'cinema_load'], check=True)
    subprocess.run(['sudo', '-u', 'postgres', 'createdb', '-p', pg_port, '-O', 'cinema_load', 'cinema_load'], check=True)
    subprocess.run(['redis-cli', 'FLUSHALL'], check=False)

    subprocess.run(['bash', './mvnw', '-B', '-DskipTests', 'clean', 'package'], check=True)
    jar = [j for j in (repo / 'target').glob('*.jar') if not j.name.endswith('.original')][0]
    env = {**os.environ, 'SERVER_ADDRESS': '127.0.0.1', 'SERVER_PORT': '8080', 'BASE_URL': 'http://127.0.0.1:8080',
           'SEED_PROFILE': 'quick' if QUICK_SMOKE_MODE else 'kaggle', 'FIXTURES': str(work_root / f'fixtures-{label}.json')}
    log_fp = open(work_root / f'api-{label}.log', 'w')
    api = subprocess.Popen(['java', *jvm_flags, '-jar', str(jar)], stdout=log_fp, stderr=subprocess.STDOUT, env=env)
    try:
        for _ in range(120):
            try:
                if urllib.request.urlopen('http://127.0.0.1:8080/actuator/health/readiness', timeout=2).status == 200: break
            except Exception: time.sleep(1)
        else:
            raise RuntimeError(pathlib.Path(log_fp.name).read_text()[-6000:])
        subprocess.run(['node', 'load_test_v2/seed.mjs'], env=env, check=True)
        cmd = ['python3', '-B', 'load_test_v2/run-phase2-benchmark.py', '--label', label, '--scenarios', *SCENARIOS,
               '--repeats', '1' if QUICK_SMOKE_MODE else '3', '--warmup', '5' if QUICK_SMOKE_MODE else '15',
               '--duration', '15' if QUICK_SMOKE_MODE else '60', '--fixtures', env['FIXTURES'],
               '--api-pids', str(api.pid), '--database-pid', str(database_pid), '--redis-pid', str(redis_pid)]
        if strict: cmd.append('--strict-phase2')
        subprocess.run(cmd, env=env, check=True)
        src = sorted((repo / 'load_test_v2/results').glob(f'*-{label}'))[-1]
        dst = work_root / 'phase2_results' / src.name
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        return dst
    finally:
        api.terminate(); api.wait(timeout=15); log_fp.close()


In [ ]:
# Ô 6 — Thực thi đo Before (và After nếu có)
before_dir = run_variant(BEFORE_REF, 'phase1-before', ['-Xms256m', '-Xmx768m'], strict=False)
after_dir = run_variant(AFTER_REF, 'phase2-after', ['-XX:+UseContainerSupport', '-Xms512m', '-Xmx1024m', '-XX:+UseG1GC'], strict=True) if AFTER_REF else None


In [ ]:
# Ô 7 — Xuất báo cáo đối chứng & nén ZIP tải về
if after_dir:
    subprocess.run(['python3', '-B', str(suite_backup / 'compare-results.py'), '--before', str(before_dir), '--after', str(after_dir),
                    '--output-md', str(work_root / 'phase2_results/comparison-report.md'), '--output-csv', str(work_root / 'phase2_results/comparison-summary.csv')], check=True)
zip_out = shutil.make_archive(str(work_root / 'cinema-phase2-kaggle-evidence'), 'zip', root_dir=str(work_root), base_dir='phase2_results')
print('Tải gói kết quả tại Output:', zip_out)
